# Behrer, Park, Wagner, Golja & Keith (2021)

Illinois county-level LOCA2 indicators. Keep this notebook and `loca2_common.py` in the same folder. Open Jupyter in that folder and run the cells from top to bottom. Each notebook uses its own kernel.

Existing files are skipped (`OVERWRITE = False`). County aggregation uses cosine-latitude area correction.

This workflow requires both relative-humidity inputs as well as temperature and precipitation.

Go to loca2_common.py to set your input and output path. 

Indicators calculated: No of days per year with Maximum Temperature in Specific Bins

## 1. Import shared functions

Required packages: numpy, xarray, dask, netCDF4, cftime, pandas, geopandas, regionmask, and tqdm. Use your existing Keeling environment.

In [1]:
from pathlib import Path
import numpy as np
import xarray as xr
import loca2_common as common
from loca2_common import (
    aggregate_to_illinois_counties,
    annual_day_count,
    to_fahrenheit,
)

## 2. Paper settings

In [2]:
PAPER_NAME = 'Behrer, Park, Wagner, Golja & Keith (2021)'
REQUIRED_VARIABLES = ('tasmax',)
OVERWRITE = False

## 3. Check paths and available runs

This cell lists the workload and loads county boundaries. It does not write climate output.

In [3]:
output_directory = common.PAPER_OUTPUT_ROOT / common.paper_directory_name(PAPER_NAME)
jobs = common.discover_paper_runs(REQUIRED_VARIABLES)
existing = sum(
    (output_directory / (
        f"{common.safe_token(PAPER_NAME)}_"
        f"{common.safe_token(model)}_{common.safe_token(member)}_{scenario}.nc"
    )).exists()
    for model, member, scenario in jobs
)
print("Input root:", common.PAPER_INPUT_ROOT)
print("Output folder:", output_directory)
print("Eligible runs:", len(jobs))
print("Existing files to skip:", existing if not OVERWRITE else 0)
print("Runs to process:", len(jobs) - existing if not OVERWRITE else len(jobs))
if not jobs:
    raise FileNotFoundError("No eligible runs found. Check input paths and required variables.")
counties = common.load_counties()
print("Counties:", len(counties))

Input root: /data/keeling/a/tahsina2/b/Climate_Impact/Data/LOCA2_Illinois
Output folder: /data/keeling/a/tahsina2/b/Climate_Impact/Data/LOCA2_Illinois/temp/Behrer, Park, Wagner, Golja & Keith (2021)
Eligible runs: 156
Existing files to skip: 0
Runs to process: 156
Counties: 102


## 4. Paper indicator calculation

In [4]:
def calculate_behrer_et_al(inputs):
    tmax_f = to_fahrenheit(inputs['tasmax'])
    result = xr.Dataset({'tmax_days_80_85F': annual_day_count((tmax_f >= 80.0) & (tmax_f < 85.0)), 'tmax_days_85_90F': annual_day_count((tmax_f >= 85.0) & (tmax_f < 90.0)), 'tmax_days_ge90F': annual_day_count(tmax_f >= 90.0)})
    result['tmax_days_80_85F'].attrs.update(units='days', frequency='annual', long_name='annual number of days with daily maximum temperature from 80 to less than 85 degrees Fahrenheit', lower_threshold_fahrenheit=80.0, upper_threshold_fahrenheit=85.0, lower_threshold_inclusive='yes', upper_threshold_inclusive='no', temperature_statistic='daily maximum temperature')
    result['tmax_days_85_90F'].attrs.update(units='days', frequency='annual', long_name='annual number of days with daily maximum temperature from 85 to less than 90 degrees Fahrenheit', lower_threshold_fahrenheit=85.0, upper_threshold_fahrenheit=90.0, lower_threshold_inclusive='yes', upper_threshold_inclusive='no', temperature_statistic='daily maximum temperature')
    result['tmax_days_ge90F'].attrs.update(units='days', frequency='annual', long_name='annual number of days with daily maximum temperature at or above 90 degrees Fahrenheit', lower_threshold_fahrenheit=90.0, lower_threshold_inclusive='yes', temperature_statistic='daily maximum temperature')
    result.attrs.update(temporal_resolution='annual', temperature_units='degrees Fahrenheit', temperature_statistic='daily maximum temperature')
    result = aggregate_to_illinois_counties(dataset=result, counties=counties)
    result.attrs.update(spatial_resolution='Illinois county', indicator_interpretation='county-area-weighted average annual number of days', weighting='grid-cell area correction only; no LandScan, population, or crop weighting')
    return result

## 5. Generate output

Existing matching filenames are skipped. A skipped file is not recalculated or checked for completeness.

In [ ]:
common.run_paper(
    paper_name=PAPER_NAME,
    required_variables=REQUIRED_VARIABLES,
    calculation_function=calculate_behrer_et_al,
    overwrite=OVERWRITE,
)

Behrer, Park, Wagner, Golja & Keith (2021):   0%|          | 0/156 [00:00<?, ?run/s]

WROTE: /data/keeling/a/tahsina2/b/Climate_Impact/Data/LOCA2_Illinois/temp/Behrer, Park, Wagner, Golja & Keith (2021)/Behrer_Park_Wagner_Golja_Keith_2021__ACCESS-CM2_r1i1p1f1_ssp245.nc


## 6. Inspect an output file

In [ ]:
files = sorted(output_directory.glob(f"{common.safe_token(PAPER_NAME)}_*.nc"))
if files:
    with xr.open_dataset(files[0], decode_timedelta=False) as dataset:
        print("Example:", files[0].name)
        print("Dimensions:", dict(dataset.sizes))
        print("Variables:", list(dataset.data_vars))
        print("Spatial resolution:", dataset.attrs.get("spatial_resolution"))
        assert dataset.sizes.get("county") == 102, "Output is not a 102-county dataset."
else:
    print("No output files found. Review the processing messages above.")